<b>Mixing/blending problem: Portfolio design</b>

In [1]:
# Input data
Bonds = {'A', 'B', 'C', 'D', 'E'}
Type = {'A' : 'Muni', 'B' : 'Agency', 'C' : 'Govn', 'D' : 'Govn', 'E' : 'Muni',}
Quality = {'A' : 2, 'B' : 2, 'C' : 1, 'D' : 1, 'E' : 5}
Maturity = {'A' : 9, 'B' : 15, 'C' : 4, 'D' : 3, 'E' : 2}
Yield = {'A' : 0.086, 'B' : 0.054, 'C' : 0.05, 'D' : 0.044, 'E' : 0.09}

In [2]:
from docplex.mp.model import Model
mdl = Model()

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [3]:
# variables
amount = mdl.continuous_var_dict(Bonds, lb=0, name="amount")

In [4]:
# objective
objective = mdl.sum(amount[b]*Yield[b] for b in Bonds)
mdl.maximize(objective)

In [5]:
# constraints TotalInvested = 10mm
TotalInvested = mdl.sum(amount[b] for b in Bonds)
mdl.add_constraint(TotalInvested <= 10) 
mdl.add_kpi(TotalInvested, 'Total Invested')

DecisionKPI(name=Total Invested,expr=amount_B+amount_D+amount_A+amount_C+amount_E)

In [6]:
# Municipal bonds
mdl.add_constraint( mdl.sum(amount[b] for b in Bonds if Type[b]=='Muni') <= 3)

docplex.mp.LinearConstraint[](amount_A+amount_E,LE,3)

In [7]:
# Quality constraint
mdl.add_constraint( mdl.sum((Quality[b]-1.4)*amount[b] for b in Bonds) <= 0)

docplex.mp.LinearConstraint[](0.600amount_B-0.400amount_D+0.600amount_A-0.400amount_C+3.600amount_E,LE,0)

In [8]:
# Maturity constraint
mdl.add_constraint( mdl.sum((Maturity[b]-5)*amount[b] for b in Bonds) <= 0)

docplex.mp.LinearConstraint[](10amount_B-2amount_D+4amount_A-amount_C-3amount_E,LE,0)

In [9]:
mdl.solve()

docplex.mp.solution.SolveSolution(obj=0.596727,values={amount_A:2.18182,..

In [10]:
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.00173903,status='optimal')

In [11]:
mdl.print_solution()

objective: 0.597
status: OPTIMAL_SOLUTION(2)
  amount_A=2.182
  amount_C=7.364
  amount_E=0.455


In [12]:
mdl.report()

* model docplex_model1 solved with objective = 0.597
*  KPI: Total Invested = 10.000


In [13]:
# Average weighted quality
AverageQuality = sum(Quality[b]*amount[b].solution_value for b in Bonds)/sum(amount[b].solution_value for b in Bonds)
mdl.add_kpi(AverageQuality, 'Average Quality')

DecisionKPI(name=Average Quality,expr=1.4)

In [14]:
mdl.report()

* model docplex_model1 solved with objective = 0.597
*  KPI: Total Invested  = 10.000
*  KPI: Average Quality = 1.400


In [15]:
# Average weighted Maturity
AverageMaturity = sum(Maturity[b]*amount[b].solution_value for b in Bonds)/sum(amount[b].solution_value for b in Bonds)
mdl.add_kpi(AverageMaturity, 'Average Maturity')

DecisionKPI(name=Average Maturity,expr=4.999999999999999)

In [16]:
mdl.report()

* model docplex_model1 solved with objective = 0.597
*  KPI: Total Invested   = 10.000
*  KPI: Average Quality  = 1.400
*  KPI: Average Maturity = 5.000
